In [ ]:
# Single-cell multi-model fine-tune for CIFAR-100 (timm)
# Requirements: timm, torch, torchvision, tqdm
# pip install timm torch torchvision tqdm

import torch, random, numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm.auto import tqdm
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
# ===== CONFIG - edit these =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 42
model_name = "mobilevitv2_150"            # try: "vit_base_patch16_224", "swin_tiny_patch4_window7_224"
pretrained = True
num_classes = 100
img_size = None                           # if None, use model default input size when available
batch_size = 128
epochs = 10
lr = 1e-3
weight_decay = 1e-4
momentum = 0.9
freeze_backbone = True                   # if True, train only the head
save_dir = Path("ckpt_multi_model")
num_workers = 4
# ================================

# Reproducibility (basic)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

# Derive model input size from timm if not provided
if img_size is None:
    try:
        cfg = timm.models.registry.get_model_default_cfg(model_name)
        img_size = cfg["input_size"][-1] if cfg and "input_size" in cfg else 224
    except Exception:
        img_size = 224

print(f"Using model: {model_name}, input image size: {img_size}")

# CIFAR-100 normalization values
mean = (0.5071, 0.4867, 0.4408)
std  = (0.2675, 0.2565, 0.2761)
writer = SummaryWriter(log_dir=f"runs/{model_name}")

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomCrop(224, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_transform = transforms.Compose([
    transforms.Resize(int(img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# Datasets & loaders
train_ds = datasets.CIFAR100(root="./data", train=True, download=True, transform=train_transform)
val_ds   = datasets.CIFAR100(root="./data", train=False, download=True, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

# Build model
model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
model.to(device)

# Optionally freeze backbone (leave head trainable)
if freeze_backbone:
    # Attempt to detect classifier/head parameter names; keep classifier/head params trainable.
    for name, p in model.named_parameters():
        if ("head" in name) or ("classifier" in name) or ("fc" in name) or ("pre_logits" in name) or ("norm" in name and p.ndim==1 and name.endswith("bias")==False):
            p.requires_grad = True
        else:
            p.requires_grad = False
    # show what remains trainable
    trainable = [n for n, p in model.named_parameters() if p.requires_grad]
    print("Trainable parameter name samples:", trainable[:10])

# Loss & optimizer (only params with requires_grad=True)
criterion = nn.CrossEntropyLoss()
opt_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(opt_params, lr=lr, momentum=momentum, weight_decay=weight_decay, nesterov=True)

# Simple accuracy helper
def accuracy(outputs, targets):
    _, preds = outputs.max(1)
    return (preds == targets).sum().item() / targets.size(0)

# Training & validation loops (kept minimal and clear)
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0
    loop = tqdm(loader, desc="Train", leave=False)
    for images, targets in loop:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs = targets.size(0)
        running_loss += loss.item() * bs
        running_correct += (outputs.argmax(1) == targets).sum().item()
        running_total += bs
        loop.set_postfix(loss=running_loss / running_total, acc=100.0 * running_correct / running_total)

    return running_loss / running_total, 100.0 * running_correct / running_total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_total = 0
    with torch.no_grad():
        loop = tqdm(loader, desc="Val", leave=False)
        for images, targets in loop:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, targets)

            bs = targets.size(0)
            running_loss += loss.item() * bs
            running_correct += (outputs.argmax(1) == targets).sum().item()
            running_total += bs
            loop.set_postfix(loss=running_loss / running_total, acc=100.0 * running_correct / running_total)

    return running_loss / running_total, 100.0 * running_correct / running_total

# Training loop
save_dir.mkdir(parents=True, exist_ok=True)
best_val_acc = 0.0
for epoch in range(1, epochs + 1):
    print(f"\nEpoch {epoch}/{epochs}")
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    # TensorBoard logging
    writer.add_scalar("Loss/Train", train_loss, epoch)
    writer.add_scalar("Loss/Val", val_loss, epoch)
    writer.add_scalar("Accuracy/Train", train_acc, epoch)
    writer.add_scalar("Accuracy/Val", val_acc, epoch)
    writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)
    # Save best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": epoch,
            "val_acc": val_acc,
            "model_name": model_name,
        }, save_dir / f"best_{model_name}.pth")
    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.2f}%")
    print(f"Val   loss: {val_loss:.4f} | Val   acc: {val_acc:.2f}% | Best: {best_val_acc:.2f}%")

print("\nDone.")
